# Task 9: Anchor-Free Object Detection (YOLOv8-Style) Loss Backpropagation

## Objective

Implement a simplified anchor-free object detection pipeline using PyTorch, OpenCV, NumPy, and Matplotlib.

The implementation includes:

- Anchor-free grid-based predictions
- Bounding-box regression
- Complete IoU (CIoU) loss
- Classification loss
- Dynamic target-to-grid assignment
- Combined multi-task detection loss
- Gradient backpropagation
- Bounding-box visualization

Unlike anchor-based detectors, each grid location directly predicts an object without predefined anchor boxes.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import cv2
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"

# ------------------------------------------------------------
# Synthetic image and ground-truth objects
# ------------------------------------------------------------

H, W = 256, 256
GRID = 16
STRIDE = H // GRID
NUM_CLASSES = 3

image = np.zeros(
    (H, W, 3),
    dtype=np.uint8
)

# Ground-truth boxes: [x1, y1, x2, y2]
targets = torch.tensor([
    [35, 45, 95, 125],
    [145, 70, 225, 155],
    [70, 165, 130, 230]
], dtype=torch.float32)

target_labels = torch.tensor([
    0, 1, 2
], dtype=torch.long)

# Draw objects for visualization
for box, label in zip(
    targets.numpy(),
    target_labels.numpy()
):

    x1, y1, x2, y2 = box.astype(int)

    cv2.rectangle(
        image,
        (x1, y1),
        (x2, y2),
        (255, 255, 255),
        2
    )

    cv2.putText(
        image,
        f"Class {label}",
        (x1, max(15, y1 - 5)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (255, 255, 255),
        1
    )

print("Image:", image.shape)
print("Ground-truth boxes:", targets)
print("Classes:", target_labels)

plt.figure(figsize=(6, 6))
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.title("Synthetic Detection Dataset")
plt.axis("off")
plt.show()

# Anchor-Free Grid and Dynamic Assignment

Each grid cell represents a candidate object location.

For every ground-truth object, the grid cell containing its center is selected.

A simple dynamic assignment score is then used:

\[
Score = P_{class}\times IoU
\]

The candidate with the highest score is selected for the target.

This demonstrates the basic idea of dynamically matching network predictions to target objects without predefined anchor boxes.

In [ ]:
# ============================================================
# Anchor-free grid
# ============================================================

def box_iou(box1, box2):

    x1 = torch.maximum(box1[..., 0], box2[..., 0])
    y1 = torch.maximum(box1[..., 1], box2[..., 1])

    x2 = torch.minimum(box1[..., 2], box2[..., 2])
    y2 = torch.minimum(box1[..., 3], box2[..., 3])

    inter = (
        torch.clamp(x2 - x1, min=0)
        * torch.clamp(y2 - y1, min=0)
    )

    area1 = (
        torch.clamp(
            box1[..., 2] - box1[..., 0],
            min=0
        )
        *
        torch.clamp(
            box1[..., 3] - box1[..., 1],
            min=0
        )
    )

    area2 = (
        torch.clamp(
            box2[..., 2] - box2[..., 0],
            min=0
        )
        *
        torch.clamp(
            box2[..., 3] - box2[..., 1],
            min=0
        )
    )

    return inter / (
        area1 + area2 - inter + 1e-7
    )


def create_grid():

    centers = []

    for gy in range(GRID):
        for gx in range(GRID):

            centers.append([
                (gx + 0.5) * STRIDE,
                (gy + 0.5) * STRIDE
            ])

    return torch.tensor(
        centers,
        dtype=torch.float32
    )


grid_centers = create_grid()

print("Grid cells:", len(grid_centers))
print("Grid center shape:", grid_centers.shape)


# ------------------------------------------------------------
# Dynamic target assignment
# ------------------------------------------------------------

def dynamic_assign(
    predicted_boxes,
    predicted_cls,
    gt_boxes,
    gt_labels
):

    assignments = []

    class_probs = torch.sigmoid(
        predicted_cls
    )

    for gt_box, gt_label in zip(
        gt_boxes,
        gt_labels
    ):

        ious = box_iou(
            predicted_boxes,
            gt_box.unsqueeze(0)
        )

        cls_scores = class_probs[
            :,
            gt_label
        ]

        scores = ious * cls_scores

        # Only candidates whose centers are inside
        # the ground-truth box
        inside = (
            (grid_centers[:, 0] >= gt_box[0])
            &
            (grid_centers[:, 0] <= gt_box[2])
            &
            (grid_centers[:, 1] >= gt_box[1])
            &
            (grid_centers[:, 1] <= gt_box[3])
        )

        scores = scores.clone()

        scores[~inside] = -1

        best_idx = torch.argmax(scores)

        assignments.append(
            (best_idx.item(), gt_box, gt_label.item())
        )

    return assignments

# CIoU Bounding Box Regression Loss

The Complete IoU loss considers:

1. Overlap between predicted and target boxes
2. Distance between their center points
3. Difference in aspect ratio

This makes CIoU more informative than IoU alone for bounding-box regression.

In [ ]:
def ciou_loss(pred_boxes, target_boxes, eps=1e-7):

    # --------------------------------------------------------
    # Intersection
    # --------------------------------------------------------

    inter_x1 = torch.maximum(
        pred_boxes[:, 0],
        target_boxes[:, 0]
    )

    inter_y1 = torch.maximum(
        pred_boxes[:, 1],
        target_boxes[:, 1]
    )

    inter_x2 = torch.minimum(
        pred_boxes[:, 2],
        target_boxes[:, 2]
    )

    inter_y2 = torch.minimum(
        pred_boxes[:, 3],
        target_boxes[:, 3]
    )

    inter_w = torch.clamp(
        inter_x2 - inter_x1,
        min=0
    )

    inter_h = torch.clamp(
        inter_y2 - inter_y1,
        min=0
    )

    intersection = inter_w * inter_h

    # --------------------------------------------------------
    # Areas and IoU
    # --------------------------------------------------------

    area_p = (
        torch.clamp(
            pred_boxes[:, 2] - pred_boxes[:, 0],
            min=0
        )
        *
        torch.clamp(
            pred_boxes[:, 3] - pred_boxes[:, 1],
            min=0
        )
    )

    area_t = (
        torch.clamp(
            target_boxes[:, 2] - target_boxes[:, 0],
            min=0
        )
        *
        torch.clamp(
            target_boxes[:, 3] - target_boxes[:, 1],
            min=0
        )
    )

    union = area_p + area_t - intersection

    iou = intersection / (
        union + eps
    )

    # --------------------------------------------------------
    # Center distance
    # --------------------------------------------------------

    p_cx = (
        pred_boxes[:, 0]
        + pred_boxes[:, 2]
    ) / 2

    p_cy = (
        pred_boxes[:, 1]
        + pred_boxes[:, 3]
    ) / 2

    t_cx = (
        target_boxes[:, 0]
        + target_boxes[:, 2]
    ) / 2

    t_cy = (
        target_boxes[:, 1]
        + target_boxes[:, 3]
    ) / 2

    center_distance = (
        (p_cx - t_cx) ** 2
        +
        (p_cy - t_cy) ** 2
    )

    # --------------------------------------------------------
    # Smallest enclosing box diagonal
    # --------------------------------------------------------

    c_x1 = torch.minimum(
        pred_boxes[:, 0],
        target_boxes[:, 0]
    )

    c_y1 = torch.minimum(
        pred_boxes[:, 1],
        target_boxes[:, 1]
    )

    c_x2 = torch.maximum(
        pred_boxes[:, 2],
        target_boxes[:, 2]
    )

    c_y2 = torch.maximum(
        pred_boxes[:, 3],
        target_boxes[:, 3]
    )

    diagonal = (
        (c_x2 - c_x1) ** 2
        +
        (c_y2 - c_y1) ** 2
        + eps
    )

    # --------------------------------------------------------
    # Aspect ratio term
    # --------------------------------------------------------

    pw = torch.clamp(
        pred_boxes[:, 2] - pred_boxes[:, 0],
        min=eps
    )

    ph = torch.clamp(
        pred_boxes[:, 3] - pred_boxes[:, 1],
        min=eps
    )

    tw = torch.clamp(
        target_boxes[:, 2] - target_boxes[:, 0],
        min=eps
    )

    th = torch.clamp(
        target_boxes[:, 3] - target_boxes[:, 1],
        min=eps
    )

    v = (
        4 / np.pi**2
    ) * (
        torch.atan(tw / th)
        -
        torch.atan(pw / ph)
    ) ** 2

    alpha = v / (
        1 - iou + v + eps
    )

    ciou = (
        iou
        - center_distance / diagonal
        - alpha * v
    )

    return 1 - ciou

# Multi-Task Detection Loss and Backpropagation

The detector produces:

- Bounding-box predictions
- Classification logits

The final loss is:

\[
L_{total}
=
L_{box}
+
L_{cls}
\]

Binary Cross-Entropy with logits is used for classification.

The complete loss is differentiated with respect to the detector outputs to demonstrate multi-task backpropagation.

In [ ]:
# ============================================================
# Synthetic anchor-free detector outputs
# ============================================================

NUM_GRID = GRID * GRID

# ------------------------------------------------------------
# Create RAW prediction parameters as leaf tensors
# ------------------------------------------------------------

raw_boxes = torch.rand(
    NUM_GRID,
    4,
    device=device,
    requires_grad=True
)

# Convert random values into valid bounding boxes
x1 = raw_boxes[:, 0] * 220
y1 = raw_boxes[:, 1] * 220

x2 = x1 + 20 + raw_boxes[:, 2] * 35
y2 = y1 + 20 + raw_boxes[:, 3] * 35

pred_boxes = torch.stack(
    [
        x1,
        y1,
        torch.clamp(x2, max=255),
        torch.clamp(y2, max=255)
    ],
    dim=1
)

# Classification logits are already leaf tensors
pred_cls = torch.randn(
    NUM_GRID,
    NUM_CLASSES,
    device=device,
    requires_grad=True
)


# ------------------------------------------------------------
# Dynamic assignment
# ------------------------------------------------------------

assignments = dynamic_assign(
    pred_boxes.detach().cpu(),
    pred_cls.detach().cpu(),
    targets,
    target_labels
)

positive_indices = [
    a[0] for a in assignments
]

print("Positive grid indices:")
print(positive_indices)


# ------------------------------------------------------------
# Build matched targets
# ------------------------------------------------------------

matched_pred_boxes = pred_boxes[
    positive_indices
]

matched_target_boxes = torch.stack([
    a[1].to(device)
    for a in assignments
])

matched_labels = torch.tensor(
    [a[2] for a in assignments],
    device=device
)


# ------------------------------------------------------------
# CIoU regression loss
# ------------------------------------------------------------

box_loss = ciou_loss(
    matched_pred_boxes,
    matched_target_boxes
).mean()


# ------------------------------------------------------------
# Classification loss
# ------------------------------------------------------------

cls_target = torch.zeros(
    NUM_GRID,
    NUM_CLASSES,
    device=device
)

for idx, label in zip(
    positive_indices,
    matched_labels
):

    cls_target[
        idx,
        label
    ] = 1.0


classification_loss = F.binary_cross_entropy_with_logits(
    pred_cls,
    cls_target
)


# ------------------------------------------------------------
# Total multi-task loss
# ------------------------------------------------------------

total_loss = (
    box_loss
    + classification_loss
)


# ------------------------------------------------------------
# Backpropagation
# ------------------------------------------------------------

total_loss.backward()


print(
    f"Box CIoU Loss       : "
    f"{box_loss.item():.4f}"
)

print(
    f"Classification Loss : "
    f"{classification_loss.item():.4f}"
)

print(
    f"Total Loss          : "
    f"{total_loss.item():.4f}"
)

# Gradient is obtained from the LEAF tensor raw_boxes
print(
    f"Box Gradient Norm   : "
    f"{raw_boxes.grad.norm().item():.6f}"
)

print(
    f"Class Gradient Norm : "
    f"{pred_cls.grad.norm().item():.6f}"
)

In [ ]:
# ============================================================
# Visualize assigned predictions and ground-truth boxes
# ============================================================

visual = image.copy()

for i, assignment in enumerate(assignments):

    grid_idx, gt_box, label = assignment

    # Predicted box
    pbox = pred_boxes[
        grid_idx
    ].detach().cpu().numpy().astype(int)

    gx = int(
        grid_centers[grid_idx, 0]
    )

    gy = int(
        grid_centers[grid_idx, 1]
    )

    # Prediction - green
    cv2.rectangle(
        visual,
        (pbox[0], pbox[1]),
        (pbox[2], pbox[3]),
        (0, 255, 0),
        2
    )

    # Ground truth - red
    gbox = gt_box.numpy().astype(int)

    cv2.rectangle(
        visual,
        (gbox[0], gbox[1]),
        (gbox[2], gbox[3]),
        (0, 0, 255),
        2
    )

    # Assigned grid center
    cv2.circle(
        visual,
        (gx, gy),
        4,
        (255, 0, 0),
        -1
    )

plt.figure(figsize=(7, 7))

plt.imshow(
    cv2.cvtColor(
        visual,
        cv2.COLOR_BGR2RGB
    )
)

plt.title(
    "Anchor-Free Dynamic Assignment\n"
    "Green = Prediction | Red = Ground Truth | "
    "Blue = Assigned Grid Center"
)

plt.axis("off")
plt.show()

# Conclusion

An anchor-free object detection loss pipeline was successfully implemented using PyTorch, OpenCV, NumPy, and Matplotlib.

The implementation demonstrated:

- Anchor-free grid-based predictions
- Dynamic target-to-grid assignment
- Complete IoU (CIoU) bounding-box regression
- Classification loss using Binary Cross-Entropy
- Multi-task detection loss
- Gradient backpropagation through regression and classification outputs
- Visualization of assigned predictions and ground-truth boxes

The CIoU loss combines overlap, center-distance, and aspect-ratio information:

\[
L_{CIoU}=1-IoU+
\frac{\rho^2}{c^2}
+\alpha v
\]

The dynamic assignment mechanism selects suitable grid locations for each target rather than relying on predefined anchor boxes.

The experiment demonstrates the fundamental loss and assignment mechanics behind modern anchor-free object detection architectures such as YOLO-style detectors.